In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

c:\Users\siwen\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Declaring the Quen3-Coder-Next Model and Loading the Tokenizer

In [ ]:
model_name = "Qwen/Qwen3-Coder-Next"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
  model_name,
  torch_dtype="auto",
  device_map="auto"
)
print("Model loaded successfully!")

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

README Generation

In [ ]:
def generate_readme(parsed_file, model, tokenizer):
    # Check if the parsed file exists, and if so it reads it
    if not os.path.exists(parsed_file):
        print(f"File {parsed_file} does not exist.")
        return None

    with open(parsed_file, 'r', encoding='utf-8', errors='ignore') as f:
        code_content = f.read()

    # Create a prompt for the model to generate the README.md content based on the code structure and contents
    prompt = f"Generate a comprehensive README.md file based on the following repository code structure and contents:\n\n{code_content}\n\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate the README.md content using the model
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
    
    # Decode the generated output and extract the README.md content
    readme_content = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "README.md" in readme_content:
        readme_content = readme_content.split("README.md")[1].strip()
        
    return readme_content

Saving the README in llm_output directory

In [ ]:
def save_readme(output_path, readme_content, prefix=""):
    # Saves the generated README.md content to a file in the specified output path
    filename = f"{prefix}generated_README.md" if prefix else "generated_README.md"
    output_path = os.path.join(output_path, filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_path}")
    return output_path

Generating the README from a repository

In [ ]:
parsed_file = "out\\apache_cordova-plugin-splashscreen_parsed_code.txt"
output_path = "llm_output"

readme = generate_readme(parsed_file, model, tokenizer)
save_readme(output_path, readme)